# 05b ML Fixed Embeddings

This notebook mirrors `05_ML_GNN_Embeddings.ipynb` but uses the fixed embedding datasets produced by `03b_fixembeddings.ipynb`.

| Dataset | Dim | What changed |
|---|---|---|
| `graphsage_fixed_srisk_dataset.parquet` | 32 | Reconstruction loss instead of link prediction; 32-dim bottleneck |
| `node2vec_fixed_srisk_dataset.parquet` | 32 | 32-dim purely structural (was 64); warm-starting for temporal coherence |

Both models are **graph-only** — no financial features added — keeping the comparison against classical centrality measures fair.

> **Note:** run `03b_fixembeddings.ipynb` first to generate the parquet files.

In [1]:
from pathlib import Path
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

from src.models.ml_train_and_store import (
    ModelTrainer,
    load_gnn_dataset,
    make_pipeline,
)

pd.set_option('display.max_columns', 200)
PROJECT_ROOT = Path().resolve().parents[1]
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


## Load Datasets

In [2]:
# Fixed GraphSAGE: 32-dim reconstruction-trained embeddings
df_sage, feature_cols_sage = load_gnn_dataset(
    PROJECT_ROOT,
    target_col="log_systemic_risk_label",
    filename="graphsage_fixed_srisk_dataset.parquet",
)
print(f"GraphSAGE fixed:  {df_sage.shape}  —  {len(feature_cols_sage)} embedding cols (32-dim)")

# Node2Vec fixed: 32-dim purely structural embeddings
df_n2v, feature_cols_n2v = load_gnn_dataset(
    PROJECT_ROOT,
    target_col="log_systemic_risk_label",
    filename="node2vec_fixed_srisk_dataset.parquet",
)
print(f"Node2Vec fixed:   {df_n2v.shape}  —  {len(feature_cols_n2v)} embedding cols (32-dim)")

GraphSAGE fixed:  (145536, 37)  —  32 embedding cols (32-dim)
Node2Vec fixed:   (145536, 37)  —  32 embedding cols (32-dim)


In [3]:
trainer_sage = ModelTrainer(
    df=df_sage,
    feature_cols=feature_cols_sage,
    target_col="log_systemic_risk_label",
)

trainer_n2v = ModelTrainer(
    df=df_n2v,
    feature_cols=feature_cols_n2v,
    target_col="log_systemic_risk_label",
)

print("GraphSAGE fixed  —", trainer_sage.train_df.shape, trainer_sage.val_df.shape, trainer_sage.test_df.shape)
print("Node2Vec enriched —", trainer_n2v.train_df.shape, trainer_n2v.val_df.shape, trainer_n2v.test_df.shape)

GraphSAGE fixed  — (109152, 37) (18192, 37) (13644, 37)
Node2Vec enriched — (109152, 37) (18192, 37) (13644, 37)


## Define Models

In [4]:
candidate_models = {
    "linear_regression": make_pipeline(LinearRegression(), scale_features=True),
    "ridge":             make_pipeline(Ridge(alpha=1.0), scale_features=True),
    "random_forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "xgboost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42), scale_features=True),
}

list(candidate_models)

['linear_regression', 'ridge', 'random_forest', 'xgboost']

## Train — Fixed GraphSAGE

In [5]:
trainer_sage.train_all(candidate_models)

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,random_forest,0.006006,0.129821,0.121977,0.033576,0.354567,0.298949,0.963855,-1.252823,-1.211295
1,xgboost,0.011602,0.162925,0.13549,0.05398,0.397317,0.324009,0.906576,-1.828809,-1.597551
2,linear_regression,0.050143,0.18192,0.113078,0.154467,0.50086,0.25803,0.23499,-3.495334,-0.647374
3,ridge,0.050196,0.309518,0.077771,0.153664,0.839892,0.203605,0.242926,-11.64085,-0.025718


In [6]:
trainer_sage.leaderboard()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,random_forest,0.006006,0.129821,0.121977,0.033576,0.354567,0.298949,0.963855,-1.252823,-1.211295
1,xgboost,0.011602,0.162925,0.13549,0.05398,0.397317,0.324009,0.906576,-1.828809,-1.597551
2,linear_regression,0.050143,0.18192,0.113078,0.154467,0.50086,0.25803,0.23499,-3.495334,-0.647374
3,ridge,0.050196,0.309518,0.077771,0.153664,0.839892,0.203605,0.242926,-11.64085,-0.025718


In [7]:
trainer_sage.test_predictions().head(20)

,bank_id,year,quarter,period,log_systemic_risk_label,prediction,abs_error
0,796,2023,1,2023Q1,0.693147,2.642224,1.949077
1,1889,2023,1,2023Q1,0.693147,2.628372,1.935225
2,2474,2023,1,2023Q1,0.693147,2.626549,1.933402
3,1957,2023,1,2023Q1,0.693147,2.626549,1.933402
4,2958,2023,1,2023Q1,0.693147,2.626549,1.933402
5,2501,2023,1,2023Q1,0.693147,2.626549,1.933402
6,3406,2023,1,2023Q1,0.693147,2.626549,1.933402
7,3218,2023,1,2023Q1,0.693147,2.626549,1.933402
8,266,2023,1,2023Q1,0.693147,2.626549,1.933402
9,4189,2023,1,2023Q1,0.693147,2.618076,1.924929


## Train — Enriched Node2Vec

In [8]:
trainer_n2v.train_all(candidate_models)

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,random_forest,0.006772,0.032153,0.030127,0.032031,0.131996,0.114281,0.967105,0.687785,0.676853
1,xgboost,0.013355,0.030482,0.026345,0.062539,0.136432,0.112552,0.8746,0.666447,0.686555
2,linear_regression,0.058798,0.080441,0.07496,0.144037,0.197041,0.167245,0.334815,0.304266,0.307922
3,ridge,0.058785,0.080419,0.074938,0.144037,0.197046,0.167243,0.334815,0.304231,0.307932


In [9]:
trainer_n2v.leaderboard()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,random_forest,0.006772,0.032153,0.030127,0.032031,0.131996,0.114281,0.967105,0.687785,0.676853
1,xgboost,0.013355,0.030482,0.026345,0.062539,0.136432,0.112552,0.8746,0.666447,0.686555
2,linear_regression,0.058798,0.080441,0.07496,0.144037,0.197041,0.167245,0.334815,0.304266,0.307922
3,ridge,0.058785,0.080419,0.074938,0.144037,0.197046,0.167243,0.334815,0.304231,0.307932


In [10]:
trainer_n2v.test_predictions().head(20)

,bank_id,year,quarter,period,log_systemic_risk_label,prediction,abs_error
0,2,2023,1,2023Q1,3.713572,2.065521,1.648051
1,8,2023,3,2023Q3,3.637586,2.064287,1.573299
2,53,2023,2,2023Q2,2.397895,0.877706,1.520189
3,878,2023,1,2023Q1,0.693147,2.211546,1.518399
4,17,2023,2,2023Q2,3.637586,2.150226,1.487360
5,6,2023,2,2023Q2,3.610918,2.129710,1.481207
6,4,2023,2,2023Q2,3.555348,2.076551,1.478797
7,926,2023,1,2023Q1,0.693147,2.136369,1.443222
8,3,2023,1,2023Q1,2.772589,1.367747,1.404842
9,2,2023,2,2023Q2,3.465736,2.089362,1.376374


## Best Models

In [11]:
print("GraphSAGE fixed best:  ", trainer_sage.best_name())
print("Node2Vec enriched best:", trainer_n2v.best_name())

GraphSAGE fixed best:   random_forest
Node2Vec enriched best: random_forest


## Comparison with Original Embeddings

Reference results from `05_ML_GNN_Embeddings.ipynb` (XGBRegressor, no warm-starting):

| Model | Train R² | Val R² | Test R² |
|---|---|---|---|
| GraphSAGE original (link prediction, 64-dim) | 0.869 | 0.073 | 0.062 |
| Node2Vec original (64-dim) | 0.876 | 0.247 | 0.302 |
| Classical features (DebtRank, PageRank, ...) | 0.946 | 0.878 | **0.900** |

Fill in below once you run this notebook:

In [12]:
import pandas as pd

def best_row(trainer, label):
    row = trainer.leaderboard().iloc[0]
    return {
        "Model": label,
        "Best estimator": row["model"],
        "Train R²": round(float(row["train_r2"]), 3),
        "Val R²":   round(float(row["validation_r2"]), 3),
        "Test R²":  round(float(row["test_r2"]), 3),
    }

comparison = pd.DataFrame([
    {"Model": "GraphSAGE original (link pred, 64-dim)", "Best estimator": "XGBRegressor", "Train R²": 0.869, "Val R²": 0.073, "Test R²": 0.062},
    {"Model": "Node2Vec original (64-dim)",             "Best estimator": "XGBRegressor", "Train R²": 0.876, "Val R²": 0.247, "Test R²": 0.302},
    {"Model": "Classical features",                     "Best estimator": "XGBRegressor", "Train R²": 0.946, "Val R²": 0.878, "Test R²": 0.900},
    best_row(trainer_sage, "GraphSAGE fixed (reconstruction, 32-dim)"),
    best_row(trainer_n2v,  "Node2Vec fixed (structural, 32-dim)"),
])

comparison.set_index("Model")

,Best estimator,Train R²,Val R²,Test R²
Model,,,,
"GraphSAGE original (link pred, 64-dim)",XGBRegressor,0.869,0.073,0.062
Node2Vec original (64-dim),XGBRegressor,0.876,0.247,0.302
Classical features,XGBRegressor,0.946,0.878,0.900
"GraphSAGE fixed (reconstruction, 32-dim)",random_forest,0.964,-1.253,-1.211
"Node2Vec fixed (structural, 32-dim)",random_forest,0.967,0.688,0.677
